En este dataset (recolectado en escuelas de Portugal), las siglas corresponden a dos colegios secundarios específicos:

GP: Gabriel Pereira (Escola Secundária Gabriel Pereira, ubicada en Évora).

MS: Mousinho da Silveira (Escola Secundária Mousinho da Silveira, ubicada en Portalegre).

In [1]:
# En este dataset (recolectado en escuelas de Portugal), las siglas corresponden a dos colegios secundarios especi­ficos:
#
# GP: Gabriel Pereira (Escola Secundaria Gabriel Pereira, ubicada en Evora).
#
# MS: Mousinho da Silveira (Escola Secundaria Mousinho da Silveira, ubicada en Portalegre).

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import shapiro
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson
from scipy import stats
from scipy.stats import randint, uniform
from sklearn.preprocessing import MinMaxScaler, RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression
import joblib
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
from datetime import datetime
import os
from pathlib import Path
from google.colab import drive
from sklearn.model_selection import train_test_split, KFold, cross_validate, RandomizedSearchCV
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.pipeline import Pipeline
import shap


# XGBoost y LightGBM son opcionales: si no estan instalados, el script
# sigue funcionando (se salteda esos modelos con un aviso) en lugar
# de romper todo el pipeline.
try:
    from xgboost import XGBRegressor
    XGBOOST_DISPONIBLE = True
except ImportError:
    XGBOOST_DISPONIBLE = False

try:
    from lightgbm import LGBMRegressor
    LIGHTGBM_DISPONIBLE = True
except ImportError:
    LIGHTGBM_DISPONIBLE = False




# 1. Montar Google Drive
drive.mount('/content/drive')

# 2. Nombre del cuaderno actual (para localizar la raÃ­z del proyecto)
NOMBRE_CUADERNO = "Trabajo Final DataII_RobertoDavidAlcoba.ipynb"

def inicializar_proyecto(nombre_cuaderno=NOMBRE_CUADERNO):
    """
    Detecta automaticamente la rai­z del proyecto a partir del cuaderno
    y crea un diccionario con todas las rutas principales.
    """
    # Buscar la carpeta rai­z donde esta el cuaderno
    ruta_raiz = None
    for path in Path('/content/drive/MyDrive').rglob(nombre_cuaderno):
        ruta_raiz = path.parent
        break

    if ruta_raiz is None:
        raise FileNotFoundError(f"No se encontra el cuaderno '{nombre_cuaderno}' en Google Drive.")

    # Definir el mapa de carpetas
    paths = {
        'root': str(ruta_raiz),
        'data': str(ruta_raiz / "data"),
        'doc': str(ruta_raiz / "doc"),
        'img': str(ruta_raiz / "img"),
        'outputs': str(ruta_raiz / "outputs"),
        'scripts': str(ruta_raiz / "scripts")
    }

    # Crear carpetas si no existen por algun motivo
    for carpeta in paths.values():
        os.makedirs(carpeta, exist_ok=True)

    # Nos posicionamos en la rai­z del proyecto
    os.chdir(paths['root'])
    print(f" Proyecto configurado en: {paths['root']}")
    return paths


# ============================================================
# 1. CARGA Y PREPARACION INICIAL DEL DATASET
# ============================================================

def cargar_dataset(ruta_archivo):
    """
    Carga el dataset y renombra las variables.
    """

    try:
        df = pd.read_csv(ruta_archivo)

        print("Archivo cargado con Exito!\n")

        # Renombrar variables
        df = df.rename(
        columns={
            'school': 'escuela',  # "GP" (Gabriel Pereira) o "MS" (Mousinho da Silveira)
            'sex': 'genero',  # "F" (Femenino) o "M" (Masculino)
            'age': 'edad',  # Numérico (de 15 a 22 años)
            'address': 'domicilio',  # "U" (Urbano) o "R" (Rural)
            'famsize': 'tamano_familia',  # "LE3" (<=3 miembros) o "GT3" (>3 miembros)
            'Pstatus': 'estado_convivencia_padres',  # "T" (Juntos/Together) o "A" (Separados/Apart)
            'Medu':    'educacion_madre',  # 0: Ninguna, 1: Primaria (4º), 2: 5º a 9º grado, 3: Secundaria, 4: Educación Superior
            'Fedu':    'educacion_padre',  # 0: Ninguna, 1: Primaria (4º), 2: 5º a 9º grado, 3: Secundaria, 4: Educación Superior
            'Mjob':    'madre_trabaja',  # "teacher", "health", "services", "at_home", "other"
            'Fjob':    'padre_trabaja',  # "teacher", "health", "services", "at_home", "other"
            'reason':  'razon_eleccion_escuela',  # home (cercanía), reputation, course, other
            'guardian': 'tutor_legal',  # "mother", "father" u "other"
            'traveltime': 'tiempo_viaje_colegio',  # 1: <15 min, 2: 15-30 min, 3: 30 min-1,  4: >1 hora
            'studytime':  'tiempo_estudio_semanal',  # 1: <2 horas, 2: 2-5 horas, 3: 5-10 horas, 4: >10 horas
            'failures': 'nro_materias_reprobadas',  # Numérico: n si 1<=n<3, si no 4
            'schoolsup': 'apoyo_educativo_extra',  # "yes" o "no"
            'famsup': 'apoyo_educativo_familiar',  # "yes" o "no"
            'paid': 'clases_particulares_pagadas',  # "yes" o "no"
            'activities': 'actividades_extracurriculares',  # "yes" o "no"
            'nursery': 'asistio_a_guarderia',  # "yes" o "no"
            'higher': 'quiere_cursar_educacion_superior',  # "yes" o "no"
            'internet': 'internet_en_casa',  # "yes" o "no"
            'romantic': 'relacion_amorosa',  # "yes" o "no"
            'famrel':   'calidad_relacion_familiar',  # Numérico del 1 (muy mala) al 5 (excelente)
            'freetime': 'tiempo_libre',  # Numérico del 1 (muy bajo) al 5 (muy alto)
            'goout': 'salidas_con_amigos',  # Numérico del 1 (muy poco) al 5 (mucho)
            'Dalc':  'alcohol_dias_laboral',  # Numérico del 1 (muy bajo) al 5 (muy alto)
            'Walc':  'alcohol_fin_semana',  # Numérico del 1 (muy bajo) al 5 (muy alto)
            'health':'estado_salud',  # Numérico del 1 (muy malo) al 5 (muy bueno)
            'absences': 'ausencias',  # Numérico (conteo de faltas, de 0 a 93)
            'G1': 'nota_periodo1',  # Numérico (de 0 a 20)
            'G2': 'nota_periodo2',  # Numérico (de 0 a 20)
            'G3': 'nota_final'  # Numérico (de 0 a 20)
        }
        )
        return df

    except FileNotFoundError:
        print(f"Error: No se encontro el archivo '{ruta_archivo}'.")
        return None

    except Exception as e:
        print(f"Ocurrio un error: {e}")
        return None

# ============================================================
# 1.1 ANALISIS UNIVARIADO DEL TARGET (guia, seccion 2.3)
# ============================================================

def analisis_univariado_target(df, paths=None, variable_objetivo='nota_final',
                                 variables_adicionales=None):
    """
    Analisis univariado exploratorio: histograma + boxplot + asimetria
    (skew) y curtosis del target, y opcionalmente de otras variables
    numericas clave. Permite decidir si conviene una transformacion
    (por ejemplo logaritmica) antes de modelar.

    Un |skew| > 1 se considera asimetria fuerte; entre 0.5 y 1,
    moderada; menor a 0.5, aproximadamente simetrica.
    """

    print("\n==========================================")
    print("A.2) ANALISIS UNIVARIADO DEL TARGET")
    print("==========================================")

    variables = [variable_objetivo] + (variables_adicionales or [])
    variables = [v for v in variables if v in df.columns]

    if not variables:
        print("  No se encontraron variables validas para analizar.")
        return {}

    fig, ejes = plt.subplots(len(variables), 2, figsize=(11, 4.5 * len(variables)))
    ejes = np.array(ejes).reshape(len(variables), 2)

    resultados_skew = {}

    for i, columna in enumerate(variables):
        datos = df[columna].dropna()
        skew = datos.skew()
        kurt = datos.kurt()
        resultados_skew[columna] = {'skew': skew, 'kurtosis': kurt}

        # Histograma + KDE
        sns.histplot(datos, kde=True, ax=ejes[i, 0], color='steelblue')
        ejes[i, 0].set_title(f'Distribucion de {columna}\nskew={skew:.2f} | kurtosis={kurt:.2f}')

        # Boxplot
        sns.boxplot(x=datos, ax=ejes[i, 1], color='lightsalmon')
        ejes[i, 1].set_title(f'Boxplot de {columna}')

        print(f"\nVariable: {columna}")
        print(f"  Media: {datos.mean():.2f} | Mediana: {datos.median():.2f} | Desvio: {datos.std():.2f}")
        print(f"  Skewness: {skew:.3f}")
        print(f"  Kurtosis: {kurt:.3f}")

        if abs(skew) > 1:
            interpretacion = "asimetria FUERTE"
        elif abs(skew) > 0.5:
            interpretacion = "asimetria MODERADA"
        else:
            interpretacion = "aproximadamente simetrica"
        print(f"  Interpretacion: {interpretacion}")

        if columna == variable_objetivo and abs(skew) > 1 and (datos > 0).all():
            print(
                "  Sugerencia: dado el sesgo fuerte y que todos los valores "
                "son positivos, podria evaluarse una transformacion "
                "logaritmica (np.log1p) del target antes de modelar."
            )
        elif columna == variable_objetivo:
            print(
                "  El target no muestra sesgo fuerte; no hace falta "
                "transformacion antes de modelar."
            )

    plt.tight_layout()

    if paths is not None:
        ruta_figura = os.path.join(paths['img'], 'analisis_univariado_target.png')
        plt.savefig(ruta_figura)
        print(f"\nGrafico de analisis univariado guardado en: {ruta_figura}")

    plt.show()
    plt.close(fig)

    return resultados_skew


# ============================================================
# 2. IDENTIFICACION DE VALORES ATIPICOS
# ============================================================

def identificar_valores_atipicos(df, paths=None):
    """
    Identifica valores ati­picos utilizando tres metodos complementarios:
    - Boxplots (inspeccion visual)
    - Z-score (distancia respecto a la media en desvi­os estandar)
    - Rango intercuartilico (IQR)
    """

    print("\n==========================================")
    print("A) IDENTIFICACION DE VALORES ATIPICOS")
    print("==========================================")

    # Variables numericas donde tiene sentido buscar outliers
    variables = [
        'edad',
        'ausencias',
        'nota_periodo1',
        'nota_periodo2',
        'nota_final'
    ]

    variables = [col for col in variables if col in df.columns]

    resultados = {}

    # --------------------------------------------------------
    # A.1) Boxplots
    # --------------------------------------------------------

    print("\n--- A.1) Boxplots ---")

    n_cols = 3
    n_filas = int(np.ceil(len(variables) / n_cols))

    fig, ejes = plt.subplots(n_filas, n_cols, figsize=(5 * n_cols, 4 * n_filas))
    ejes = np.array(ejes).reshape(-1)

    for i, columna in enumerate(variables):
        sns.boxplot(y=df[columna], ax=ejes[i], color='skyblue')
        ejes[i].set_title(f'Boxplot: {columna}')

    # Ocultar ejes sobrantes si el grid no se llena completo
    for j in range(len(variables), len(ejes)):
        ejes[j].axis('off')

    plt.tight_layout()

    if paths is not None:
        ruta_figura = os.path.join(paths['img'], 'boxplots_outliers.png')
        plt.savefig(ruta_figura)
        print(f"Boxplots guardados en: {ruta_figura}")

    plt.show()
    plt.close(fig)

    # --------------------------------------------------------
    # A.2) Z-score e IQR (por variable)
    # --------------------------------------------------------

    for columna in variables:

        datos = df[columna].dropna()

        # --- Z-score ---
        # |z| > 3 se considera atÃ­pico (criterio estandar)
        z_scores = np.abs(stats.zscore(datos))
        outliers_zscore = datos[z_scores > 3]

        # --- IQR ---
        Q1 = df[columna].quantile(0.25)
        Q3 = df[columna].quantile(0.75)

        IQR = Q3 - Q1

        limite_inferior = Q1 - 1.5 * IQR
        limite_superior = Q3 + 1.5 * IQR

        valores_atipicos_iqr = df[
            (df[columna] < limite_inferior) |
            (df[columna] > limite_superior)
        ]

        resultados[columna] = {
            'outliers_zscore': len(outliers_zscore),
            'outliers_iqr': len(valores_atipicos_iqr),
            'Q1': Q1,
            'Q3': Q3,
            'IQR': IQR,
            'limite_inferior': limite_inferior,
            'limite_superior': limite_superior
        }

        print(f"\nVariable: {columna}")
        print(f"  Media: {datos.mean():.2f} | Desvi­o estandar: {datos.std():.2f}")
        print(f"  Cantidad de valores ati­picos (Z-score > 3): {len(outliers_zscore)}")
        print(f"  Q1: {Q1:.2f}")
        print(f"  Q3: {Q3:.2f}")
        print(f"  IQR: {IQR:.2f}")
        print(f"  Li­mite inferior: {limite_inferior:.2f}")
        print(f"  Li­mite superior: {limite_superior:.2f}")
        print(f"  Cantidad de valores ati­picos (IQR): {len(valores_atipicos_iqr)}")

    return df, resultados


# ============================================================
# 2.B REVISION DE CONSISTENCIA DE CATEGORIAS
# ============================================================

def revisar_consistencia_categorias(df):
    """
    Revisa las variables categoricas (tipo texto) en busca de
    inconsistencias: errores de tipeo, espacios en blanco sobrantes
    o diferencias de mayusculas/minusculas para una misma categoria.
    """

    print("\n==========================================")
    print("REVISION DE CONSISTENCIA DE CATEGORIAS")
    print("==========================================")

    columnas_categoricas = df.select_dtypes(include='object').columns

    inconsistencias_encontradas = {}

    for columna in columnas_categoricas:

        valores_originales = df[columna].dropna().unique()

        # Normalizamos: quitamos espacios y pasamos todo a minÃºsculas
        valores_normalizados = (
            df[columna]
            .dropna()
            .astype(str)
            .str.strip()
            .str.lower()
            .unique()
        )

        print(f"\nVariable: {columna}")
        print(f"  Valores originales ({len(valores_originales)}): "
              f"{sorted(valores_originales.tolist())}")

        if len(valores_normalizados) < len(valores_originales):
            inconsistencias_encontradas[columna] = {
                'originales': sorted(valores_originales.tolist()),
                'normalizados': sorted(valores_normalizados.tolist())
            }
            print("  ?? Posible inconsistencia detectada "
                  "(mayusculas/minusculas o espacios en blanco distintos "
                  "para la misma categori­a).")
        else:
            print("  ? Sin inconsistencias evidentes de formato.")

    if not inconsistencias_encontradas:
        print("\nNo se detectaron inconsistencias de categorÃ­as en el dataset.")
    else:
        print(
            f"\nSe detectaron inconsistencias en "
            f"{len(inconsistencias_encontradas)} variable(s): "
            f"{list(inconsistencias_encontradas.keys())}"
        )

    return inconsistencias_encontradas


# ============================================================
# 3. CONVERSIÓN DE VARIABLES CATEGÓRICAS
# ============================================================

def convertir_variables_categoricas(df):
    """
    Convierte variables categóricas binarias a valores numéricos.
    """

    print("\n==========================================")
    print("B) CONVERSIÓN DE VARIABLES CATEGÓRICAS")
    print("==========================================")

    # Variables binarias yes/no
    variables_binarias = [
        'apoyo_educativo_extra',
        'apoyo_educativo_familiar',
        'clases_particulares_pagadas',
        'actividades_extracurriculares',
        'asistio_a_guarderia',
        'quiere_cursar_educacion_superior',
        'internet_en_casa',
        'relacion_amorosa'
    ]

    for columna in variables_binarias:

        if columna in df.columns:

            df[columna] = df[columna].map({
                'yes': 1,
                'no': 0
            })

            print(f"{columna}: yes/no → 1/0")

    # Género
    if 'genero' in df.columns:
        df['genero'] = df['genero'].map({
            'M': 1,
            'F': 0
        })

        print("genero: M/F → 1/0")

    # Domicilio
    if 'domicilio' in df.columns:
        df['domicilio'] = df['domicilio'].map({
            'U': 1,
            'R': 0
        })

        print("domicilio: U/R → 1/0")

    # Estado de convivencia de los padres
    if 'estado_convivencia_padres' in df.columns:
        df['estado_convivencia_padres'] = df[
            'estado_convivencia_padres'
        ].map({
            'T': 1,
            'A': 0
        })

        print("estado_convivencia_padres: T/A → 1/0")

    return df


# ============================================================
# 3.B CODIFICACIÓN DE VARIABLES NOMINALES (ONE-HOT)
# ============================================================

def codificar_variables_nominales(df, columnas=None, drop_first=True):
    """
    Codifica mediante one-hot encoding las variables categóricas
    nominales (sin orden intrínseco) que quedan como texto luego de
    convertir_variables_categoricas().

    Sin este paso, columnas como 'escuela', 'tamano_familia',
    'madre_trabaja', 'padre_trabaja', 'razon_eleccion_escuela' y
    'tutor_legal' quedan tipo object y son descartadas SILENCIOSAMENTE
    por select_dtypes(include=['number']) en seleccionar_variables_relevantes(),
    es decir, nunca llegan a competir en el SelectKBest.

    Parámetros
    ----------
    df : DataFrame ya procesado por convertir_variables_categoricas()
         y codificar_variables_ordinales().
    columnas : lista opcional de columnas a codificar. Si es None,
        se usa un listado por defecto con las nominales conocidas
        de este dataset.
    drop_first : si True, elimina la primera categoría de cada
        variable para evitar multicolinealidad perfecta (la trampa
        de las variables dummy / dummy variable trap). Recomendado
        si luego se va a ajustar una regresión lineal.

    Notas
    -----
    - pd.get_dummies() devuelve columnas tipo bool, no int. select_dtypes
      (include=['number']) tampoco reconoce bool como numérico, así que
      acá se castean explícitamente a int para que después SÍ entren en
      el SelectKBest.
    - Columnas ya binarias como 'tamano_familia' (GT3/LE3) también
      quedan cómodamente representadas con una sola dummy (drop_first=True).
    """

    print("\n==========================================")
    print("B.2) CODIFICACION DE VARIABLES NOMINALES (ONE-HOT)")
    print("==========================================")

    if columnas is None:
        columnas = [
            'escuela',
            'tamano_familia',
            'madre_trabaja',
            'padre_trabaja',
            'razon_eleccion_escuela',
            'tutor_legal'
        ]

    columnas_existentes = [c for c in columnas if c in df.columns]
    columnas_faltantes = [c for c in columnas if c not in df.columns]

    if columnas_faltantes:
        print(
            f"  Aviso: no se encontraron en el DataFrame las columnas "
            f"{columnas_faltantes}, se omiten."
        )

    if not columnas_existentes:
        print("  No hay columnas nominales para codificar.")
        return df

    columnas_previas = set(df.columns)

    df = pd.get_dummies(
        df,
        columns=columnas_existentes,
        drop_first=drop_first
    )

    columnas_nuevas = [c for c in df.columns if c not in columnas_previas]

    # get_dummies devuelve bool; lo casteamos a int para que
    # select_dtypes(include=['number']) las reconozca más adelante.
    df[columnas_nuevas] = df[columnas_nuevas].astype(int)

    print(f"Variables codificadas (one-hot, drop_first={drop_first}):")
    for columna in columnas_existentes:
        print(f" - {columna}")

    print(f"\nColumnas dummy creadas ({len(columnas_nuevas)}):")
    for columna in columnas_nuevas:
        print(f" - {columna}")

    return df

# ============================================================
# 4. CODIFICACION DE VARIABLES ORDINALES
# ============================================================

def codificar_variables_ordinales(df):
    """
    Convierte variables ordinales a valores numÃ©ricos manteniendo
    el orden de sus categorÃ­as.
    """

    print("\n==========================================")
    print("C) CODIFICACION DE VARIABLES ORDINALES")
    print("==========================================")

    # En este dataset varias variables ya estan codificadas
    # numericamente. Se convierten a tipo entero.

    variables_ordinales = [
        'educacion_madre',
        'educacion_padre',
        'tiempo_viaje_colegio',
        'tiempo_estudio_semanal',
        'nro_materias_reprobadas',
        'calidad_relacion_familiar',
        'tiempo_libre',
        'salidas_con_amigos',
        'alcohol_dias_laboral',
        'alcohol_fin_semana',
        'estado_salud'
    ]

    for columna in variables_ordinales:

        if columna in df.columns:
            df[columna] = pd.to_numeric(
                df[columna],
                errors='coerce'
            ).astype('Int64')

            print(f"{columna}: codificacion ordinal conservada")

    return df


# ============================================================
# 5. CREACION DE NUEVAS VARIABLES
# ============================================================

def crear_nuevas_variables(df):
    """
    Crea variables derivadas a partir de las variables originales.
    """

    print("\n==========================================")
    print("D) CREACION DE NUEVAS VARIABLES")
    print("==========================================")

    # Promedio de las notas de los dos primeros peri­odos:
    # resume el desempeño del alumno en los dos primeros peri­odos en un solo numero,
    # antes de que exista la nota final. Es util como variable predictora de nota_final
    df['promedio_periodos'] = (df['nota_periodo1'] + df['nota_periodo2']) / 2
    print("Creada: promedio_periodos")

    # Evolucion entre el primer y segundo peri­odo:
    # captura la tendencia del alumno, no solo su nivel. Un alumno que va de 8 a 14 tiene la misma "nota promedio"
    # que uno que va de 14 a 8, pero son situaciones completamente distintas (uno mejora, otro empeora)
    # Esta variable te permite distinguir alumnos que estan progresando de los que estan retrocediendo
    df['evolucion_academica'] = (df['nota_periodo2'] - df['nota_periodo1'] )
    print("Creada: evolucion_academica")

    # Promedio general de las tres notas
    # ojo con esta — incluye nota_final, que es la variable objetivo.
    # Esta variable solo tiene sentido solo como una medida descriptiva/exploratoria del rendimiento global del alumno
    # a lo largo del año ===> NO ENTRA COMO FEATURE
    df['promedio_notas'] = (df['nota_periodo1'] + df['nota_periodo2'] + df['nota_final']) / 3
    print("Creada: promedio_notas")

    # Indicador de apoyo educativo:
    # combina dos fuentes de apoyo (el que da la escuela + el que da la familia) en un solo Ã­ndice de 0 a 2
    # Como ambas son binarias (0/1), sumarlas te da una escala simple: 0 = sin apoyo de ningun lado, 1 = apoyo de una fuente,
    # 2 = apoyo de ambas. Sirve para testear la hipotesis de que "mas apoyo educativo ? mejor rendimiento" sin tener que meter dos variables separadas y correlacionadas al modelo
    df['indice_apoyo_educativo'] = (df['apoyo_educativo_extra'] + df['apoyo_educativo_familiar'])
    print("Creada: indice_apoyo_educativo")

    # Indicador de consumo de alcohol:
    # unifica el consumo de alcohol en dÃ­a de semana y fin de semana en un solo indicador de "consumo promedio general".
    # Es util porque estas dos variables suelen estar correlacionadas entre si­ (multicolinealidad), y condensarlas en una reduce redundancia
    # en el modelo, ademas de dar una medida mas estable del habito de consumo del alumno.
    df['promedio_alcohol'] = (df['alcohol_dias_laboral'] + df['alcohol_fin_semana']) / 2
    print("Creada: promedio_alcohol")

    # Indicador de rendimiento academico:
    # convierte el problema en una clasificacion binaria (alto rendimiento / no) ademas de tenerlo como regresion (nota_final continua, 0-20)
    # El corte en 15 (sobre 20) suele usarse como umbral de "buen alumno"
    df['rendimiento_alto'] = np.where(df['nota_final'] >= 15,1,0)
    print("Creada: rendimiento_alto")

    #1. Índice de riesgo académico — combina las señales de riesgo que ya tenés (reprobadas, ausencias, poco estudio)
    df['indice_riesgo_academico'] = (df['nro_materias_reprobadas'] * 2 +   # peso mayor, es la señal más fuerte
    (df['ausencias'] > df['ausencias'].median()).astype(int) - df['tiempo_estudio_semanal'])
    print("Creada: indice_riesgo_academico")

    #2.Nivel educativo del hogar — promedia educación de ambos padres (cada una individualmente es débil, pero juntas dan una señal de entorno más estable)
    df['nivel_educativo_parental'] = (df['educacion_madre'] + df['educacion_padre']) / 2
    print("Creada: nivel_educativo_parental")

    #3.Índice de compromiso académico — combina estudio, apoyo recibido e intención de seguir estudiando
    df['indice_compromiso_academico'] = (df['tiempo_estudio_semanal'] +df['indice_apoyo_educativo'] +df['quiere_cursar_educacion_superior'])
    print("Creada: indice_compromiso_academico")

    #4.Índice de riesgo social — agrupa salidas, alcohol y relación amorosa
    df['indice_riesgo_social'] = (df['salidas_con_amigos'] + df['promedio_alcohol'] + df['relacion_amorosa'])
    print("Creada: indice_riesgo_social")


    return df


# ============================================================
# 5.B ANÃLISIS BIVARIADO / MULTIVARIADO
# ============================================================

def analisis_bivariado_multivariado(
    df,
    paths=None,
    variable_objetivo='nota_final',
    top_n=40,
    excluir_del_analisis=None
):
    """
    Realiza el analisis bivariado/multivariado exploratorio:

    1) Matriz de correlacion (heatmap) entre variables numericas y el target.
    2) Scatter plots de las variables numericas mas correlacionadas
       con el target.
    3) Boxplots de variables categoricas vs. target.
    4) Deteccion de multicolinealidad entre predictoras (VIF), clave
       antes de ajustar un modelo de regresion lineal.

    Parametros
    ----------
    df : DataFrame contiene todos los campos en numericos, para que las
    correlaciones tengan sentido.
    paths : dict con las rutas del proyecto (para guardar las figuras).
    variable_objetivo : nota_final.
    top_n : 40 lista las variables correlacionadas a graficar.
    excluir_del_analisis : columnas que no deben entrar como predictoras
    (por ejemplo variables "fuga" que ya contienen al target, como
        'promedio_notas', 'rendimiento_alto','promedio_periodos').
    Ademas se debe excluir las variables originales que estan incluidas en las
    nuevas variables
    """

    print("\n==========================================")
    print("G) ANALISIS BIVARIADO / MULTIVARIADO")
    print("==========================================")

    if excluir_del_analisis is None:
        excluir_del_analisis = ['rendimiento_alto',
                                'promedio_notas',
                                'nota_periodo1',
                                'nota_periodo2',
                                'evolucion_academica',
                                'promedio_periodos',
                                'alcohol_dias_laboral',
                                'alcohol_fin_semana',
                                'apoyo_educativo_extra',
                                'apoyo_educativo_familiar',
                                'nro_materias_reprobadas',
                                'ausencias',
                                'tiempo_estudio_semanal',
                                'educacion_madre',
                                'educacion_padre',
                                'tiempo_estudio_semanal',
                                'quiere_cursar_educacion_superior',
                                'salidas_con_amigos',
                                'relacion_amorosa',
                                'indice_apoyo_educativo',
                                'promedio_alcohol',
                                'padre_trabaja_other',
                                'madre_trabaja_other']
    resultados = {}

    # --------------------------------------------------------
    # G.1) Matriz de correlacion (heatmap)
    # --------------------------------------------------------

    print("\n--- G.1) Matriz de correlacion (numericas + target) ---")

    df_numerico = df.select_dtypes(include=['number']).copy()

    # Sacamos del heatmap las variables "fuga" para no distorsionar
    # la lectura, pero dejamos el target.
    columnas_excluir_heatmap = [
        c for c in excluir_del_analisis if c != variable_objetivo
    ]
    df_numerico = df_numerico.drop(
        columns=columnas_excluir_heatmap, errors='ignore'
    )

    matriz_corr = df_numerico.corr()
    resultados['matriz_correlacion'] = matriz_corr

    plt.figure(figsize=(max(10, 0.5 * len(matriz_corr.columns)),
                         max(8, 0.5 * len(matriz_corr.columns))))
    sns.heatmap(
        matriz_corr,
        annot=False,
        cmap='coolwarm',
        center=0,
        linewidths=0.3
    )
    plt.title('Matriz de correlacion - Variables numericas y target')
    plt.tight_layout()

    if paths is not None:
        ruta_figura = os.path.join(paths['img'], 'heatmap_correlacion.png')
        plt.savefig(ruta_figura)
        print(f"Heatmap de correlacion guardado en: {ruta_figura}")

    plt.show()
    plt.close()

    # --------------------------------------------------------
    # G.2) Scatter plots de las variables mas correlacionadas
    #      con el target
    # --------------------------------------------------------

    print(f"\n--- G.2) Scatter plots vs. '{variable_objetivo}' ---")

    if variable_objetivo not in matriz_corr.columns:
        print(
            f"  La variable objetivo '{variable_objetivo}' no es "
            "numerica o no esta; en el DataFrame. Se omite este paso."
        )
    else:
        correlaciones_target = (
            matriz_corr[variable_objetivo]
            .drop(labels=[variable_objetivo], errors='ignore')
            .dropna()
        )

        top_variables = (
            correlaciones_target
            .abs()
            .sort_values(ascending=False)
            .head(top_n)
            .index
            .tolist()
        )

        resultados['top_correlacionadas_con_target'] = (
            correlaciones_target.loc[top_variables]
            .sort_values(ascending=False)
        )

        print("Variables numericas mas correlacionadas con el target:")
        print(resultados['top_correlacionadas_con_target'])
        ahora = datetime.now()
        df_correlacion = resultados['top_correlacionadas_con_target'].reset_index()
        df_correlacion.columns = ['Variable', 'Correlacion']
        nombre_archivo = ahora.strftime("correlacion_vs_target_%d%m%Y_%H%M%S.csv")
        ruta_salida = os.path.join(paths['outputs'], nombre_archivo)
        df_correlacion.to_csv(ruta_salida, index=False)
        print(f"\nSe genero con existo el archivo correlacion_vs_target.csv ")

        n_cols = 3
        n_filas = int(np.ceil(len(top_variables) / n_cols)) if top_variables else 1

        fig, ejes = plt.subplots(
            n_filas, n_cols, figsize=(5 * n_cols, 4 * n_filas)
        )
        ejes = np.array(ejes).reshape(-1)

        for i, columna in enumerate(top_variables):
            sns.regplot(
                x=df[columna],
                y=df[variable_objetivo],
                ax=ejes[i],
                scatter_kws={'alpha': 0.4, 's': 15},
                line_kws={'color': 'red'}
            )
            corr_val = correlaciones_target[columna]
            ejes[i].set_title(f'{columna} vs. {variable_objetivo}\n(r = {corr_val:.2f})')

        for j in range(len(top_variables), len(ejes)):
            ejes[j].axis('off')

        plt.tight_layout()

        if paths is not None:
            ruta_figura = os.path.join(paths['img'], 'scatter_vs_target.png')
            plt.savefig(ruta_figura)
            print(f"Scatter plots guardados en: {ruta_figura}")

        plt.show()
        plt.close(fig)

    # --------------------------------------------------------
    # G.3) Boxplots de variables categoricas vs. target
    # --------------------------------------------------------

    print(f"\n--- G.3) Boxplots de categoricas vs. '{variable_objetivo}' ---")

    columnas_categoricas = df.select_dtypes(include='object').columns.tolist()

    if variable_objetivo not in df.columns:
        print(
            f"  La variable objetivo '{variable_objetivo}' no esta "
            "en el DataFrame. Se omite este paso."
        )
    elif not columnas_categoricas:
        print("  No se encontraron variables categoricas (tipo texto) "
              "para comparar contra el target.")
    else:
        n_cols = 3
        n_filas = int(np.ceil(len(columnas_categoricas) / n_cols))

        fig, ejes = plt.subplots(
            n_filas, n_cols, figsize=(5 * n_cols, 4 * n_filas)
        )
        ejes = np.array(ejes).reshape(-1)

        for i, columna in enumerate(columnas_categoricas):
            sns.boxplot(
                x=df[columna],
                y=df[variable_objetivo],
                ax=ejes[i]
            )
            ejes[i].set_title(f'{variable_objetivo} segun {columna}')
            ejes[i].tick_params(axis='x', rotation=45)

        for j in range(len(columnas_categoricas), len(ejes)):
            ejes[j].axis('off')

        plt.tight_layout()

        if paths is not None:
            ruta_figura = os.path.join(
                paths['img'], 'boxplots_categoricas_vs_target.png'
            )
            plt.savefig(ruta_figura)
            print(f"Boxplots categoricas vs. target guardados en: {ruta_figura}")

        plt.show()
        plt.close(fig)

    # --------------------------------------------------------
    # G.4) Multicolinealidad entre predictoras (VIF)
    # --------------------------------------------------------

    print("\n--- G.4) Multicolinealidad entre predictoras (VIF) ---")

    columnas_excluir_vif = list(set(excluir_del_analisis + [variable_objetivo]))

    X_vif = df.select_dtypes(include=['number']).drop(
        columns=columnas_excluir_vif, errors='ignore'
    )

    # Algunas columnas pueden venir como Int64 (entero "nullable" de pandas),
    # que es un extension dtype. statsmodels necesita arrays de NumPy en
    # float64 puro, asi­ que convertimos expli­citamente.
    X_vif = X_vif.astype('float64')

    # VIF requiere datos completos (sin NaN)
    X_vif = X_vif.dropna()

    if X_vif.shape[1] < 2:
        print("  No hay suficientes variables numericas predictoras "
              "para calcular VIF.")
        resultados['vif'] = None
    else:
        # --- VIF CON constante (estandar: agrega intercepto al regresar
        # cada Xi contra las demas). Es el VIF clasico, centrado en la
        # media, y el que corresponde citar como criterio de referencia
        # (>5 moderado, >10 severo). ---
        X_vif_const = sm.add_constant(X_vif)
        vif_con_constante = pd.Series(
            [
                variance_inflation_factor(X_vif_const.values, i)
                for i in range(X_vif_const.shape[1])
            ],
            index=X_vif_const.columns,
        ).drop('const')

        # --- VIF SIN constante (regresion sin intercepto, "uncentered").
        # Suele dar valores mas altos porque no separa la varianza
        # explicada por la media; se calcula ademas por pedido explicito. ---
        vif_sin_constante = pd.Series(
            [
                variance_inflation_factor(X_vif.values, i)
                for i in range(X_vif.shape[1])
            ],
            index=X_vif.columns,
        )

        vif_data = pd.DataFrame({
            'variable': X_vif.columns,
            'VIF_Constante': vif_con_constante.reindex(X_vif.columns).values,
            'VIF_Sin_Constante': vif_sin_constante.reindex(X_vif.columns).values,
        })
        vif_data = vif_data.sort_values(by='VIF_Constante', ascending=False)

        # ---------------------------------------------------------
        # GUARDAR VIF EN CSV
        # ---------------------------------------------------------
        ahora = datetime.now()
        os.makedirs("outputs", exist_ok=True)
        nombre_archivo = ahora.strftime("VIF_%d%m%Y_%H%M%S.csv")
        ruta_vif = os.path.join("outputs",nombre_archivo)
        vif_data.to_csv(ruta_vif,index=False,encoding="utf-8-sig")
        print(f"\nArchivo VIF guardado en: {ruta_vif}")

        # Guardar también en resultados
        resultados['vif'] = vif_data
        print("Factor de Inflacion de la Varianza (VIF Constante y VIF Sin Constante) por variable:")
        print(vif_data.to_string(index=False))


        print(
            "\nCriterio orientativo: VIF > 5 sugiere multicolinealidad "
            "moderada; VIF > 10 sugiere multicolinealidad severa "
            "(la variable es altamente explicada por las demas "
            "predictoras y conviene revisarla/eliminarla antes de "
            "una regresion lineal)."
        )
        variables_perfectas = vif_data[
            (vif_data['VIF_Constante'] > 1000) | (vif_data['VIF_Constante'].isna())
        ]['variable'].tolist()

        variables_alta_multicolinealidad = vif_data[
            (vif_data['VIF_Constante'] > 10) & (vif_data['VIF_Constante'] <= 1000)
        ]['variable'].tolist()

        if variables_perfectas:
            print(
                f"\nMulticolinealidad PERFECTA (VIF extremo, "
                f"probable combinacion lineal exacta con otra variable): "
                f"{variables_perfectas}\n"
                "   Esto suele pasar con variables derivadas de otras ya "
                "incluidas (por ejemplo 'promedio_periodos' respecto de "
                "'nota_periodo1' y 'nota_periodo2', o 'promedio_alcohol' "
                "respecto de 'alcohol_dias_laboral' y 'alcohol_fin_semana'). "
                "Para un modelo de regresiÃ³n lineal hay que elegir UNA de "
                "las dos representaciones (la original o la derivada), "
                "nunca ambas."
            )

        if variables_alta_multicolinealidad:
            print(
                f"\nLas  Variables con VIF > 10 (multicolinealidad alta, "
                f"no necesariamente exacta): {variables_alta_multicolinealidad}"
            )

        if not variables_perfectas and not variables_alta_multicolinealidad:
            print("\nNo se detectaron variables con VIF > 10.")

    return resultados


# ============================================================
# 6. NORMALIZACION
# ============================================================

def normalizar_variables(df, umbral_skew=1.0, forzar_robust=None, excluir=None):
    """
    Normaliza las variables numericas con un enfoque HIBRIDO:

    - RobustScaler (basado en mediana e IQR) para las variables con
      distribucion fuertemente asimetrica o con outliers severos
      (|skew| > umbral_skew). MinMaxScaler es muy sensible a valores
      extremos: un solo outlier en el maximo o el mi­nimo comprime a
      todos los demas casos "normales" en una porcion muy chica del
      rango [0, 1]. RobustScaler no sufre este problema porque usa
      mediana e IQR en vez de min/max.
    - MinMaxScaler (rango 0-1) para el resto de las variables, cuya
      distribucion es razonablemente simetrica y sin outliers severos.

    Nota: las variables escaladas con RobustScaler NO quedan
    necesariamente en el rango [0, 1] (pueden tener valores negativos
    o mayores a 1), a diferencia de las escaladas con MinMaxScaler.
    Esto es esperable y no es un error: es el trade-off de ser robusto
    a outliers.

    Ver tambien: el docstring de seleccionar_variables_relevantes()
    explica por que este paso de normalizacion NO afecta el ranking
    que produce SelectKBest con f_regression (ese selector es
    invariante a este tipo de escalado lineal).

    QUE VARIABLES SE NORMALIZAN
    ----------------------------
    Para cuando se llama esta funcion (Paso 9 del pipeline), TODAS las
    columnas del df ya son numericas: las categoricas binarias se
    convirtieron en el Paso 6, las ordinales en el Paso 7 y las
    nominales via one-hot en el Paso 8.C. Por eso ya no tiene sentido
    normalizar solo una lista fija de columnas: se normalizan
    dinamicamente TODAS las columnas numericas del df, EXCEPTO:

    - Las binarias (0/1), detectadas automaticamente por tener 2 o
      menos valores unicos. Esto cubre tanto las variables si/no ya
      convertidas (por ejemplo 'internet_en_casa') como TODAS las
      dummies generadas por el one-hot (por ejemplo 'escuela_MS').
      Escalar una variable 0/1 no aporta nada (ya esta en una escala
      comparable) y complica la lectura de sus coeficientes en un
      modelo lineal.
    - Las que se pasen explicitamente en el parametro 'excluir'.

    Antes, esta funcion normalizaba solo una lista fija de 9 columnas
    ('edad', 'ausencias', las notas y los promedios), dejando afuera
    variables ordinales como 'educacion_madre', 'tiempo_estudio_semanal'
    o 'salidas_con_amigos'. Eso generaba un dataset con escalas
    mezcladas (0-1 para unas, 0-5 para otras) que perjudica a modelos
    sensibles a la escala como KNN, SVR o regresion Ridge/Lasso.

    Parametros
    ----------
    umbral_skew : valor absoluto de asimetri­a (skewness) a partir del
        cual una variable se considera "fuertemente asimetrica" y se
        escala con RobustScaler en vez de MinMaxScaler.
    forzar_robust : lista opcional de columnas que se quieren forzar
        a RobustScaler sin importar su skew (por ejemplo si ya se
        sabe, por el analisis de outliers del paso A, que una
        variable tiene valores atÃ­picos relevantes).
    excluir : lista opcional de columnas numericas que NO se quieren
        normalizar aunque no sean binarias (por ejemplo un ID).
    """

    print("\n==========================================")
    print("E) NORMALIZACION DE VARIABLES (escalado hi­brido)")
    print("==========================================")

    if excluir is None:
        excluir = []

    columnas_numericas = df.select_dtypes(include=['number']).columns.tolist()

    columnas_binarias = [
        columna for columna in columnas_numericas
        if df[columna].nunique(dropna=True) <= 2
    ]

    variables_existentes = [
        columna for columna in columnas_numericas
        if columna not in columnas_binarias and columna not in excluir
    ]

    print(
        f"\nColumnas binarias detectadas automaticamente (no se "
        f"normalizan, {len(columnas_binarias)} en total):"
    )
    print(f"  {columnas_binarias}")

    if forzar_robust is None:
        forzar_robust = []

    # --------------------------------------------------------
    # Decidir, por variable, que escalador usar segun su asimetri­a
    # --------------------------------------------------------

    skews = df[variables_existentes].skew()

    print("\nAsimetri­a (skew) por variable:")
    for columna in variables_existentes:
        print(f"  {columna:25s} skew = {skews[columna]:6.2f}")

    variables_robust = [
        columna for columna in variables_existentes
        if abs(skews[columna]) > umbral_skew or columna in forzar_robust
    ]

    variables_minmax = [
        columna for columna in variables_existentes
        if columna not in variables_robust
    ]

    # --------------------------------------------------------
    # RobustScaler para variables con outliers/asimetri­a fuerte
    # --------------------------------------------------------

    if variables_robust:
        robust_scaler = RobustScaler()

        df[variables_robust] = robust_scaler.fit_transform(
            df[variables_robust]
        )

        print(
            f"\nVariables escaladas con RobustScaler "
            f"(|skew| > {umbral_skew} o forzadas manualmente):"
        )

        for columna in variables_robust:
            print(f" - {columna}")

    # --------------------------------------------------------
    # MinMaxScaler para el resto
    # --------------------------------------------------------

    if variables_minmax:
        minmax_scaler = MinMaxScaler()

        df[variables_minmax] = minmax_scaler.fit_transform(
            df[variables_minmax]
        )

        print("\nVariables escaladas con MinMaxScaler (rango 0-1):")

        for columna in variables_minmax:
            print(f" - {columna}")

    return df


# ============================================================
# 7. SELECCION DE VARIABLES RELEVANTES
# ============================================================

def seleccionar_variables_relevantes(df):
    """
    Selecciona las variables numericas mas relacionadas con
    la variable objetivo 'nota_final'.

    Se utilizan DOS criterios complementarios:

    - SelectKBest con f_regression: mide relacion LINEAL (equivalente
      a un test de significancia sobre la correlacion de Pearson).
      Es rapido y facil de interpretar, pero puede subestimar una
      variable que tenga una relacion fuerte pero NO lineal con el
      target (por ejemplo, un umbral o una relacion en forma de U).
    - mutual_info_regression (Informacion Mutua): mide cualquier tipo
      de dependencia estadistica entre X e Y, lineal o no lineal, sin
      asumir una forma funcional particular. Es mas costoso
      computacionalmente y su estimacion (basada en k-vecinos mas
      cercanos) tiene algo de aleatoriedad, por eso se fija
      random_state para que el resultado sea reproducible.

    Se reportan ambos rankings por separado: si una variable aparece
    arriba en los dos, es una candidata solida; si aparece alta en
    Informacion Mutua pero baja en f_regression, es señal de que su
    relacion con el target probablemente no es lineal y conviene
    revisarla antes de descartarla solo por su score de f_regression.

    NOTA sobre el orden del pipeline (normalizar -> seleccionar):
    ------------------------------------------------------------
    f_regression calcula, para cada variable, un estadistico F
    derivado de la correlacion de Pearson con el target. La
    correlacion de Pearson es INVARIANTE ante transformaciones
    lineales/afines (x' = a·x + b, con a > 0) de X o de Y. Tanto
    MinMaxScaler ((x-min)/(max-min)) como RobustScaler
    ((x-mediana)/IQR) son transformaciones de ese tipo.

    En la practica esto significa que el ranking de variables que
    entrega SelectKBest con f_regression es EXACTAMENTE el mismo
    se hayan normalizado las variables antes o no, y sin importar
    que escalador se haya usado (MinMax, Robust, Standard). El paso
    de normalizacion (funcion normalizar_variables) es necesario
    para otras etapas del proyecto (comparar coeficientes en una
    misma escala, o si en el futuro se prueba Ridge/Lasso, KNN,
    PCA, etc.), pero no cambia en nada el resultado de este
    selector de variables.
    """

    print("\n==========================================")
    print("F) SELECCION DE VARIABLES RELEVANTES")
    print("==========================================")

    variable_objetivo = 'nota_final'

    # Variables que NO utilizaremos como predictoras
    excluir = [
        variable_objetivo,
        'rendimiento_alto',
        'promedio_notas',
        'nota_periodo1',
        'nota_periodo2',
        'evolucion_academica',
        'promedio_periodos',
        'alcohol_dias_laboral',
        'alcohol_fin_semana',
        'apoyo_educativo_extra',
        'apoyo_educativo_familiar',
        'nro_materias_reprobadas',
        'ausencias',
        'tiempo_estudio_semanal',
        'educacion_madre',
        'educacion_padre',
        'tiempo_estudio_semanal',
        'quiere_cursar_educacion_superior',
        'salidas_con_amigos',
        'relacion_amorosa',
        'indice_apoyo_educativo',
        'promedio_alcohol',
        'padre_trabaja_other',
        'madre_trabaja_other']
    # Seleccionamos solamente variables numericas
    X = df.select_dtypes(include=['number']).drop(
        columns=excluir,
        errors='ignore'
    )

    y = df[variable_objetivo]

    # Eliminar columnas con valores nulos
    X = X.fillna(X.median())

    # Seleccionar las 10 variables mÃ¡s relevantes
    k = min(10, X.shape[1])

    selector = SelectKBest(
        score_func=f_regression,
        k=k
    )

    selector.fit(X, y)

    variables_seleccionadas = X.columns[
        selector.get_support()
    ]

    print("\nVariables seleccionadas:")

    for variable in variables_seleccionadas:
        print(f" - {variable}")

    # --------------------------------------------------------
    # Informacion Mutua (mutual_info_regression)
    # --------------------------------------------------------
    # Captura relaciones no lineales que f_regression puede pasar
    # por alto. random_state fijo para que el resultado sea
    # reproducible (el estimador usa k-vecinos mas cercanos, que
    # tiene un componente aleatorio).

    mi_scores = mutual_info_regression(X, y, random_state=42)

    variables_seleccionadas_mi = (
        pd.Series(mi_scores, index=X.columns)
        .sort_values(ascending=False)
        .head(k)
        .index
    )

    print(f"\nTop {k} variables por Informacion Mutua (mutual_info_regression):")

    for variable in variables_seleccionadas_mi:
        print(f" - {variable}")


    # Mostrar puntuacion de cada variable (f_regression + Informacion Mutua)
    resultados = pd.DataFrame({
        'variable': X.columns,
        'puntuacion_f_regression': selector.scores_,
        'puntuacion_mutual_info': mi_scores,
    })

    # Ordenar de forma descendente por f_regression
    resultados = resultados.sort_values(
        by='puntuacion_f_regression',
        ascending=False
    )

    print("\nRanking de variables (f_regression y Informacion Mutua):")

    print(resultados.to_string(index=False))

    # --------------------------------------------------------
    # Guardar archivo CSV
    # --------------------------------------------------------
    ahora = datetime.now()
    os.makedirs("outputs", exist_ok=True)
    nombre_archivo = ahora.strftime("f_regression_vs_mutual_info_regression_%d%m%Y_%H%M%S.csv")
    ruta_salida = os.path.join("outputs",nombre_archivo)
    resultados.to_csv(ruta_salida, index=False, encoding="utf-8-sig")
    print(f"\nSe genero exitosamente el archivo CSV en: {ruta_salida}")


    # Variables que aparecen en el top-k de AMBOS criterios: la
    # senal mas confiable, coincide relacion lineal y no lineal.
    variables_coincidentes = [
        v for v in variables_seleccionadas
        if v in variables_seleccionadas_mi.tolist()
    ]

    print(
        f"\nVariables en el top-{k} de AMBOS criterios "
        f"(f_regression Y mutual_info): {variables_coincidentes}"
    )

    return df, variables_seleccionadas, resultados

# ============================================================
# 8. ENTRENAMIENTO Y COMPARACION DE FAMILIAS DE MODELOS
# ============================================================
"""
Despues de ejecutar la funcion seleccionar_variables_relevantes() el dataframe cuenta
(variables numericas, normalizadas, sin nulos).

Se definen DOS conjuntos de variables (Feature Sets), pensados para
DOS familias de modelos distintas:

- Feature Set A (interpretable): las variables con mayor F-score
  (relacion LINEAL mas fuerte con nota_final). Se usa como linea base
  con modelos lineales (Linear Regression, Ridge, Lasso), donde los
  coeficientes se pueden interpretar directamente.

- Feature Set B (no lineal): Set A + variables con alto Mutual
  Information pero bajo F-score (indice_compromiso_academico,
  tutor_legal_mother, padre_trabaja_services), que probablemente
  aportan senal NO lineal que un modelo de arboles si puede aprovechar
  aunque una regresion lineal no la vea. Se usa con modelos de
  ensamble (Random Forest, XGBoost, LightGBM).

Ambas familias se evalua con:
  - Validacion cruzada (K-Fold, cv=5) sobre TODO el dataset -> metricas
    robustas, poco sensibles a como cayo el split train/test.
  - Un holdout train/test (80/20) -> metricas "de produccion" y,
    para los modelos lineales, coeficientes; para los ensambles,
    feature importances.

Metricas reportadas: R2, RMSE y MAE
"""


# ============================================================
# DEFINICION DE LOS FEATURE SETS
# ============================================================

FEATURE_SET_A = [
    'indice_riesgo_academico',
    'nivel_educativo_parental',
    'edad',
    'indice_riesgo_social',
    'tiempo_viaje_colegio',
    'madre_trabaja_health',
    'domicilio',
    'genero',
    'clases_particulares_pagadas',
    'internet_en_casa',
]

FEATURE_SET_B = FEATURE_SET_A + [
    'indice_compromiso_academico',
    'tutor_legal_mother',
    'padre_trabaja_services',
]

RANDOM_STATE = 42


# ============================================================
# 2) R2 AJUSTADO + METRICAS (REEMPLAZA a las funciones previas)
# ============================================================

def calcular_r2_ajustado(r2, n, p):
    """
    R2 ajustado = 1 - (1 - R2) * (n - 1) / (n - p - 1)

    n: cantidad de observaciones usadas para calcular el R2.
    p: cantidad de variables predictoras (features) del modelo.

    A diferencia del R2 comun, el R2 ajustado penaliza agregar
    variables que no aportan poder predictivo real, por eso es mas
    justo para comparar modelos con distinta cantidad de features
    (como el Feature Set A de 10 variables vs. el Feature Set B de 13).
    """
    denominador = n - p - 1
    if denominador <= 0:
        return np.nan
    return 1 - (1 - r2) * (n - 1) / denominador


def _r2_ajustado_scorer(p):
    """Devuelve un scorer compatible con cross_validate/scoring que
    calcula R2 ajustado en cada fold, usando el n real de ESE fold."""

    def scorer(estimator, X, y):
        y_pred = estimator.predict(X)
        r2 = r2_score(y, y_pred)
        n = len(y)
        return calcular_r2_ajustado(r2, n, p)

    return scorer


def _metricas_cv(modelo, X, y, cv, p):
    """Corre validacion cruzada y devuelve R2, R2 ajustado, RMSE y MAE
    promedio (+ desvio estandar) sobre los folds de validacion.

    p: cantidad de features de X (necesario para el R2 ajustado).
    """
    scoring = {
        'r2': 'r2',
        'r2_ajustado': _r2_ajustado_scorer(p),
        'neg_rmse': 'neg_root_mean_squared_error',
        'neg_mae': 'neg_mean_absolute_error',
    }
    resultados = cross_validate(
        modelo, X, y, cv=cv, scoring=scoring, n_jobs=-1
    )
    return {
        'R2_cv_media': resultados['test_r2'].mean(),
        'R2_cv_std': resultados['test_r2'].std(),
        'R2_ajustado_cv_media': resultados['test_r2_ajustado'].mean(),
        'R2_ajustado_cv_std': resultados['test_r2_ajustado'].std(),
        'RMSE_cv_media': -resultados['test_neg_rmse'].mean(),
        'RMSE_cv_std': resultados['test_neg_rmse'].std(),
        'MAE_cv_media': -resultados['test_neg_mae'].mean(),
        'MAE_cv_std': resultados['test_neg_mae'].std(),
    }


def _metricas_holdout(modelo, X_train, X_test, y_train, y_test, p):
    """Entrena en el train y evalua en el test holdout, incluyendo
    R2 ajustado (p = cantidad de features usadas)."""
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    n = len(y_test)
    return {
        'R2_test': r2,
        'R2_ajustado_test': calcular_r2_ajustado(r2, n, p),
        'RMSE_test': np.sqrt(mean_squared_error(y_test, y_pred)),
        'MAE_test': mean_absolute_error(y_test, y_pred),
    }, modelo

# ============================================================
# 4)  DIAGNOSTICO DE RESIDUOS
# ============================================================

def diagnostico_residuos_regresion(modelo, X_test, y_test, nombre_modelo,
                                     paths=None, tests_estadisticos=True):
    """
    Diagnostico grafico y estadistico de residuos (guia, seccion 5,
    "Diagnostico adicional"):

    - Real vs. predicho
    - Residuos vs. predichos (chequeo visual de homocedasticidad y
      patrones no capturados por el modelo)
    - Histograma + QQ-plot de residuos (chequeo visual de normalidad)
    - Test de Shapiro-Wilk (normalidad de residuos)
    - Test de Breusch-Pagan (homocedasticidad)
    - Estadistico de Durbin-Watson (independencia / autocorrelacion
      de residuos)

    Los tests estadisticos (Shapiro, Breusch-Pagan) son los supuestos
    clasicos de la regresion LINEAL: tienen sentido pleno para Linear
    Regression, Ridge y Lasso. Para modelos de arboles/ensambles esos
    supuestos no aplican de la misma manera (no son parte de su teoria),
    por eso para esos modelos se puede pasar tests_estadisticos=False
    y quedarse solo con los graficos, que siguen siendo utiles como
    diagnostico general de ajuste.
    """

    y_pred = modelo.predict(X_test)
    residuos = np.asarray(y_test) - np.asarray(y_pred)

    fig, ejes = plt.subplots(2, 2, figsize=(12, 10))

    # 1) Real vs. predicho
    ejes[0, 0].scatter(y_test, y_pred, alpha=0.5, color='steelblue')
    minimo = min(np.min(y_test), np.min(y_pred))
    maximo = max(np.max(y_test), np.max(y_pred))
    ejes[0, 0].plot([minimo, maximo], [minimo, maximo], 'r--', label='Prediccion perfecta')
    ejes[0, 0].set_xlabel('Valor real')
    ejes[0, 0].set_ylabel('Valor predicho')
    ejes[0, 0].set_title('Real vs. Predicho')
    ejes[0, 0].legend()

    # 2) Residuos vs. predichos (homocedasticidad / patrones)
    ejes[0, 1].scatter(y_pred, residuos, alpha=0.5, color='darkorange')
    ejes[0, 1].axhline(0, color='red', linestyle='--')
    ejes[0, 1].set_xlabel('Valor predicho')
    ejes[0, 1].set_ylabel('Residuo (real - predicho)')
    ejes[0, 1].set_title('Residuos vs. Predichos')

    # 3) Histograma de residuos
    sns.histplot(residuos, kde=True, ax=ejes[1, 0], color='mediumseagreen')
    ejes[1, 0].set_title('Distribucion de los residuos')
    ejes[1, 0].axvline(0, color='red', linestyle='--')

    # 4) QQ-plot (normalidad)
    stats.probplot(residuos, dist='norm', plot=ejes[1, 1])
    ejes[1, 1].set_title('QQ-plot de residuos')

    plt.suptitle(f'Diagnostico de residuos - {nombre_modelo}', fontsize=13)
    plt.tight_layout()

    if paths is not None:
        nombre_archivo = f"residuos_{nombre_modelo.replace(' ', '_').replace('(', '').replace(')', '')}.png"
        ruta_figura = os.path.join(paths['img'], nombre_archivo)
        plt.savefig(ruta_figura)
        print(f"  Grafico de residuos guardado en: {ruta_figura}")

    plt.show()
    plt.close(fig)

    resultados_tests = {}

    if tests_estadisticos:
        print(f"\n  --- Tests de supuestos ({nombre_modelo}) ---")

        # --- Shapiro-Wilk: normalidad de residuos ---
        # H0: los residuos vienen de una distribucion normal.
        stat_shapiro, p_shapiro = shapiro(residuos)
        resultados_tests['shapiro_stat'] = stat_shapiro
        resultados_tests['shapiro_pvalue'] = p_shapiro
        veredicto_shapiro = (
            "NO se rechaza normalidad (p > 0.05)" if p_shapiro > 0.05
            else "se RECHAZA normalidad (p <= 0.05)"
        )
        print(f"  Shapiro-Wilk: estadistico={stat_shapiro:.4f}, p-value={p_shapiro:.4f} -> {veredicto_shapiro}")

        # --- Breusch-Pagan: homocedasticidad ---
        # H0: varianza de los residuos es constante (homocedasticidad).
        X_test_const = sm.add_constant(np.asarray(X_test))
        bp_stat, bp_pvalue, bp_fstat, bp_fpvalue = het_breuschpagan(residuos, X_test_const)
        resultados_tests['breusch_pagan_stat'] = bp_stat
        resultados_tests['breusch_pagan_pvalue'] = bp_pvalue
        veredicto_bp = (
            "NO se rechaza homocedasticidad (p > 0.05)" if bp_pvalue > 0.05
            else "se RECHAZA homocedasticidad -> hay heterocedasticidad (p <= 0.05)"
        )
        print(f"  Breusch-Pagan: estadistico={bp_stat:.4f}, p-value={bp_pvalue:.4f} -> {veredicto_bp}")

        # --- Durbin-Watson: independencia / autocorrelacion ---
        # Rango 0-4. ~2 = sin autocorrelacion. <1.5 = autocorrelacion
        # positiva. >2.5 = autocorrelacion negativa.
        dw_stat = durbin_watson(residuos)
        resultados_tests['durbin_watson'] = dw_stat
        if 1.5 <= dw_stat <= 2.5:
            veredicto_dw = "sin evidencia relevante de autocorrelacion"
        elif dw_stat < 1.5:
            veredicto_dw = "posible autocorrelacion POSITIVA"
        else:
            veredicto_dw = "posible autocorrelacion NEGATIVA"
        print(f"  Durbin-Watson: estadistico={dw_stat:.4f} -> {veredicto_dw}")

    return resultados_tests


# ============================================================
# 3) FAMILIA 1: MODELOS LINEALES
# ============================================================

def entrenar_modelos_lineales(df, variable_objetivo, feature_set, cv, paths=None):
    """
    Entrena y compara Linear Regression, Ridge y Lasso sobre el
    Feature Set A. Ahora incluye R2 ajustado y diagnostico de
    residuos (real vs. predicho, homocedasticidad, normalidad,
    autocorrelacion) para cada modelo.
    """

    print("\n------------------------------------------------------")
    print("Familia LINEAL / INTERPRETABLE - Feature Set A")
    print("------------------------------------------------------")

    cols = [c for c in feature_set if c in df.columns]
    faltantes = [c for c in feature_set if c not in df.columns]
    if faltantes:
        print(f"  Aviso: faltan columnas {faltantes}, se excluyen.")

    X = df[cols].copy()
    y = df[variable_objetivo]
    p = X.shape[1]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE
    )

    alphas = np.logspace(-3, 3, 50)

    modelos = {
        'Linear Regression': LinearRegression(),
        'Ridge (RidgeCV)': RidgeCV(alphas=alphas, cv=cv),
        'Lasso (LassoCV)': LassoCV(alphas=alphas, cv=cv, random_state=RANDOM_STATE, max_iter=10000),
    }

    filas_resumen = []
    modelos_entrenados = {}
    coeficientes = {}
    diagnosticos = {}

    for nombre, modelo in modelos.items():
        print(f"\n>> {nombre}")

        metricas_cv = _metricas_cv(modelo, X, y, cv, p)
        metricas_holdout, modelo_ajustado = _metricas_holdout(
            modelo, X_train, X_test, y_train, y_test, p
        )
        modelos_entrenados[nombre] = modelo_ajustado

        print(
            f"   CV (5-fold)  -> R2: {metricas_cv['R2_cv_media']:.3f} | "
            f"R2 ajustado: {metricas_cv['R2_ajustado_cv_media']:.3f} | "
            f"RMSE: {metricas_cv['RMSE_cv_media']:.3f} | "
            f"MAE: {metricas_cv['MAE_cv_media']:.3f}"
        )
        print(
            f"   Holdout 20%% -> R2: {metricas_holdout['R2_test']:.3f} | "
            f"R2 ajustado: {metricas_holdout['R2_ajustado_test']:.3f} | "
            f"RMSE: {metricas_holdout['RMSE_test']:.3f} | "
            f"MAE: {metricas_holdout['MAE_test']:.3f}"
        )

        if hasattr(modelo_ajustado, 'alpha_'):
            print(f"   Alpha optimo seleccionado por CV: {modelo_ajustado.alpha_:.4f}")

        coefs = pd.Series(modelo_ajustado.coef_, index=cols).sort_values(
            key=np.abs, ascending=False
        )
        coeficientes[nombre] = coefs
        print("   Coeficientes (ordenados por magnitud):")
        for var, val in coefs.items():
            print(f"     {var:35s} {val: .4f}")

        # Diagnostico de residuos (supuestos plenos: es un modelo lineal)
        diagnosticos[nombre] = diagnostico_residuos_regresion(
            modelo_ajustado, X_test, y_test, nombre, paths=paths,
            tests_estadisticos=True
        )

        filas_resumen.append({
            'familia': 'Lineal / Interpretable',
            'feature_set': 'A',
            'modelo': nombre,
            **metricas_cv,
            **metricas_holdout,
        })

    resumen = pd.DataFrame(filas_resumen)
    return resumen, modelos_entrenados, coeficientes, diagnosticos


# ============================================================
# 3) FAMILIA 2: ENSAMBLES
#    RandomizedSearchCV para tunear hiperparametros.
# ============================================================

def entrenar_modelos_ensamble(df, variable_objetivo, feature_set, cv, paths=None, n_iter=25):
    """
    Entrena y compara Random Forest, XGBoost y LightGBM sobre el
    Feature Set B, ahora con RandomizedSearchCV para tunear
    hiperparametros (en vez de valores fijos), R2 ajustado y
    diagnostico de residuos.

    n_iter: cantidad de combinaciones de hiperparametros que prueba
    cada RandomizedSearchCV (mas alto = busqueda mas fina pero mas
    lenta).
    """

    print("\n------------------------------------------------------")
    print("Familia ENSAMBLE / NO LINEAL - Feature Set B")
    print("------------------------------------------------------")

    cols = [c for c in feature_set if c in df.columns]
    faltantes = [c for c in feature_set if c not in df.columns]
    if faltantes:
        print(f"  Aviso: faltan columnas {faltantes}, se excluyen.")

    X = df[cols].copy()
    y = df[variable_objetivo]
    p = X.shape[1]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE
    )

    espacios_busqueda = {
        'Random Forest': (
            RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
            {
                'n_estimators': randint(100, 500),
                'max_depth': [None, 3, 5, 8, 12],
                'min_samples_leaf': randint(1, 10),
                'min_samples_split': randint(2, 10),
                'max_features': ['sqrt', 'log2', None],
            },
        ),
    }

    if XGBOOST_DISPONIBLE:
        espacios_busqueda['XGBoost'] = (
            XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1),
            {
                'n_estimators': randint(100, 500),
                'max_depth': randint(2, 8),
                'learning_rate': uniform(0.01, 0.29),
                'subsample': uniform(0.6, 0.4),
                'colsample_bytree': uniform(0.6, 0.4),
                'reg_alpha': uniform(0, 1),
                'reg_lambda': uniform(0, 2),
            },
        )
    else:
        print("\n  Aviso: xgboost no esta instalado (pip install xgboost). Se omite este modelo.")

    if LIGHTGBM_DISPONIBLE:
        espacios_busqueda['LightGBM'] = (
            LGBMRegressor(random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
            {
                'n_estimators': randint(100, 500),
                'num_leaves': randint(5, 31),
                'max_depth': [-1, 3, 5, 7],
                'learning_rate': uniform(0.01, 0.29),
                'min_child_samples': randint(5, 30),
                'subsample': uniform(0.6, 0.4),
            },
        )
    else:
        print("  Aviso: lightgbm no esta instalado (pip install lightgbm). Se omite este modelo.")

    filas_resumen = []
    modelos_entrenados = {}
    importancias = {}
    diagnosticos = {}

    for nombre, (estimador_base, param_dist) in espacios_busqueda.items():
        print(f"\n>> {nombre} (RandomizedSearchCV, n_iter={n_iter})")

        busqueda = RandomizedSearchCV(
            estimator=estimador_base,
            param_distributions=param_dist,
            n_iter=n_iter,
            cv=cv,
            scoring='r2',
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        busqueda.fit(X_train, y_train)
        modelo = busqueda.best_estimator_

        print(f"   Mejores hiperparametros: {busqueda.best_params_}")
        print(f"   Mejor R2 en la busqueda (CV interna sobre train): {busqueda.best_score_:.3f}")

        # Metricas finales: CV sobre TODO el dataset y holdout, ya con
        # el modelo tuneado (mismos criterios que la familia lineal,
        # para que la tabla comparativa final sea homogenea)
        metricas_cv = _metricas_cv(modelo, X, y, cv, p)
        metricas_holdout, modelo_ajustado = _metricas_holdout(
            modelo, X_train, X_test, y_train, y_test, p
        )
        modelos_entrenados[nombre] = modelo_ajustado

        print(
            f"   CV (5-fold)  -> R2: {metricas_cv['R2_cv_media']:.3f} | "
            f"R2 ajustado: {metricas_cv['R2_ajustado_cv_media']:.3f} | "
            f"RMSE: {metricas_cv['RMSE_cv_media']:.3f} | "
            f"MAE: {metricas_cv['MAE_cv_media']:.3f}"
        )
        print(
            f"   Holdout 20%% -> R2: {metricas_holdout['R2_test']:.3f} | "
            f"R2 ajustado: {metricas_holdout['R2_ajustado_test']:.3f} | "
            f"RMSE: {metricas_holdout['RMSE_test']:.3f} | "
            f"MAE: {metricas_holdout['MAE_test']:.3f}"
        )

        imp = pd.Series(
            modelo_ajustado.feature_importances_, index=cols
        ).sort_values(ascending=False)
        importancias[nombre] = imp
        print("   Feature importances:")
        for var, val in imp.items():
            print(f"     {var:35s} {val:.4f}")

        # Diagnostico solo grafico (los supuestos de homocedasticidad/
        # normalidad son propios de regresion lineal, no de arboles)
        diagnosticos[nombre] = diagnostico_residuos_regresion(
            modelo_ajustado, X_test, y_test, nombre, paths=paths,
            tests_estadisticos=False
        )

        filas_resumen.append({
            'familia': 'Ensamble / No lineal',
            'feature_set': 'B',
            'modelo': nombre,
            **metricas_cv,
            **metricas_holdout,
        })

    resumen = pd.DataFrame(filas_resumen)
    return resumen, modelos_entrenados, importancias, diagnosticos


# ============================================================
# COMPARACION FINAL
# ============================================================

def graficar_comparacion(resumen, paths=None):
    fig, ejes = plt.subplots(1, 2, figsize=(14, 5))

    orden = resumen.sort_values('R2_test', ascending=False)
    colores = orden['familia'].map({
        'Lineal / Interpretable': '#4C72B0',
        'Ensamble / No lineal': '#DD8452',
    })

    ejes[0].barh(orden['modelo'], orden['R2_test'], color=colores)
    ejes[0].set_xlabel('R2 (holdout)')
    ejes[0].set_title('Comparacion de R2 por modelo')
    ejes[0].invert_yaxis()

    orden_rmse = resumen.sort_values('RMSE_test', ascending=True)
    colores_rmse = orden_rmse['familia'].map({
        'Lineal / Interpretable': '#4C72B0',
        'Ensamble / No lineal': '#DD8452',
    })
    ejes[1].barh(orden_rmse['modelo'], orden_rmse['RMSE_test'], color=colores_rmse)
    ejes[1].set_xlabel('RMSE (holdout)')
    ejes[1].set_title('Comparacion de RMSE por modelo')
    ejes[1].invert_yaxis()

    plt.tight_layout()

    if paths is not None:
        ruta_figura = os.path.join(paths['img'], 'comparacion_modelos.png')
        plt.savefig(ruta_figura)
        print(f"\nGrafico de comparacion guardado en: {ruta_figura}")

    plt.show()
    plt.close(fig)


def entrenar_y_comparar_familias_modelos(df, paths=None, variable_objetivo='nota_final'):
    """
    Con R2 ajustado a la tabla comparativa y
    genera el diagnostico de residuos de cada modelo (real vs.
    predicho, residuos, y para los lineales ademas Shapiro-Wilk,
    Breusch-Pagan y Durbin-Watson).
    """

    print("\n==========================================")
    print("H) ENTRENAMIENTO Y COMPARACION DE MODELOS")
    print("==========================================")

    if variable_objetivo not in df.columns:
        raise ValueError(
            f"La variable objetivo '{variable_objetivo}' no esta en el DataFrame."
        )

    # FEATURE_SET_A y FEATURE_SET_B se toman de las constantes ya definidas


    cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    resumen_a, modelos_a, coeficientes, diag_a = entrenar_modelos_lineales(
    df, variable_objetivo, FEATURE_SET_A, cv, paths=paths)

    resumen_b, modelos_b, importancias, diag_b = entrenar_modelos_ensamble(
    df, variable_objetivo, FEATURE_SET_B, cv, paths=paths)


    resumen = pd.concat([resumen_a, resumen_b], ignore_index=True)
    resumen = resumen.sort_values('R2_test', ascending=False).reset_index(drop=True)

    columnas_orden = [
        'familia', 'feature_set', 'modelo',
        'R2_cv_media', 'R2_cv_std',
        'R2_ajustado_cv_media', 'R2_ajustado_cv_std',
        'RMSE_cv_media', 'RMSE_cv_std',
        'MAE_cv_media', 'MAE_cv_std',
        'R2_test', 'R2_ajustado_test', 'RMSE_test', 'MAE_test',
    ]
    resumen = resumen[columnas_orden]

    print("\n------------------------------------------------------")
    print("TABLA COMPARATIVA FINAL (ordenada por R2 en holdout)")
    print("------------------------------------------------------")
    print(resumen.to_string(index=False))

    mejor = resumen.iloc[0]
    print(
        f"\nMejor modelo por R2 en holdout: {mejor['modelo']} "
        f"({mejor['familia']}, Feature Set {mejor['feature_set']}) "
        f"-> R2: {mejor['R2_test']:.3f} | R2 ajustado: {mejor['R2_ajustado_test']:.3f} "
        f"| RMSE: {mejor['RMSE_test']:.3f}"
    )

    ahora = datetime.now()
    carpeta_salida = paths['outputs'] if paths is not None else "outputs"
    os.makedirs(carpeta_salida, exist_ok=True)
    nombre_archivo = ahora.strftime("comparacion_modelos_%d%m%Y_%H%M%S.csv")
    ruta_salida = os.path.join(carpeta_salida, nombre_archivo)
    resumen.to_csv(ruta_salida, index=False, encoding="utf-8-sig")
    print(f"\nSe genero exitosamente el archivo CSV en: {ruta_salida}")

    graficar_comparacion(resumen, paths)

    modelos_entrenados = {**modelos_a, **modelos_b}
    extras = {
        'coeficientes': coeficientes,
        'importancias': importancias,
        'diagnosticos': {**diag_a, **diag_b},
    }

    return resumen, modelos_entrenados, extras

#==============================================================
# SELECCION DEL MEJOR MODELO PARA PREDECIR LA VARIABLE OBJETIVO
#==============================================================
def entrenar_mejor_modelo_pipeline(df, original,paths=None, variable_objetivo='nota_final'):
    # Lista de variables del Feature Set A
    feature_set_a = [
        'indice_riesgo_academico',
        'nivel_educativo_parental',
        'edad',
        'indice_riesgo_social',
        'tiempo_viaje_colegio',
        'madre_trabaja_health',
        'domicilio',
        'genero',
        'clases_particulares_pagadas',
        'internet_en_casa'
    ]

    # Separar X e y
    X_train_final = df[feature_set_a]
    y_train_final = original[variable_objetivo]


    # Preprocesador que escala las características automáticas al rango que el modelo entiende
    preprocesador = ColumnTransformer(
        transformers=[
            ('scaler', StandardScaler(), feature_set_a)
        ],
        remainder='drop'
    )

    # El Pipeline ahora escala Y LUEGO predice
    pipeline_modelo = Pipeline(steps=[
        ('preprocesamiento', preprocesador),
        ('regresion_lineal', LinearRegression())
    ])

    # Entrenar el Pipeline
    pipeline_modelo.fit(X_train_final, y_train_final)

    # Crear artefacto contenedor
    artefacto_modelo = {
        'pipeline': pipeline_modelo,
        'features': feature_set_a
    }

    # Definir carpeta de destino respetando el diccionario PATHS
    if paths is not None and 'outputs' in paths:
        carpeta_salida = paths['outputs']
    else:
        carpeta_salida = 'outputs'

    os.makedirs(carpeta_salida, exist_ok=True)
    ruta_salida_modelo = os.path.join(carpeta_salida, 'modelo_lineal_nota_final.pkl')

    # Guardar en disco
    joblib.dump(artefacto_modelo, ruta_salida_modelo)
    print(f"Pipeline entrenado y guardado con éxito en: {ruta_salida_modelo}")

    return artefacto_modelo

#==============================================================
# UNA OPCION IMPORTANTE ES APLICAR SHAP PARA PROFUNDIZAR EN EL DIAGNOSTIVO DEL MODELO Y ENTENDER
# EL COMPORTAMIENTOS QUE LOS COEFICIENTE LINEALES O FEATURE IMPORTANTES TRADICIONALES NO LO LOGRAN
# EXPLICAR DEL TODO
#==============================================================

def analisis_explicabilidad_shap(modelos_entrenados, df, feature_sets, paths=None, variable_objetivo='nota_final'):
    """
    Genera gráficos de explicación global mediante SHAP (SHapley Additive exPlanations)
    para comparar la interpretabilidad entre modelos lineales y ensambles de árboles.

    Soporta:
    - TreeExplainer para ensambles (Random Forest, XGBoost, LightGBM).
    - LinearExplainer para modelos lineales (LinearRegression, Ridge, Lasso).
    """

    print("\n==========================================")
    print("I) ANALISIS DE EXPLICABILIDAD GLOBAL (SHAP)")
    print("==========================================")

    # Recrear división train/test respetando el random_state usado en el entrenamiento
    y = df[variable_objetivo]

    os.makedirs(os.path.join(paths['img'], 'shap'), exist_ok=True) if paths else None

    for nombre_modelo, modelo in modelos_entrenados.items():

        # Determinar qué Feature Set usó este modelo
        if nombre_modelo in ['Linear Regression', 'Ridge (RidgeCV)', 'Lasso (LassoCV)']:
            features = feature_sets['A']
        else:
            features = feature_sets['B']

        cols = [c for c in features if c in df.columns]
        X = df[cols]

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=RANDOM_STATE
        )

        print(f"\n>> Generando SHAP Summary Plot para: {nombre_modelo}")

        try:
            # 1. Modelos de Árboles / Ensambles
            if nombre_modelo in ['Random Forest', 'XGBoost', 'LightGBM']:
                explainer = shap.TreeExplainer(modelo)
                shap_values = explainer.shap_values(X_test)

            # 2. Modelos Lineales
            else:
                explainer = shap.LinearExplainer(modelo, X_train)
                shap_values = explainer.shap_values(X_test)

            # Generar gráfico Summary Plot (Beeswarm)
            plt.figure(figsize=(10, 6))
            shap.summary_plot(shap_values, X_test, show=False)
            plt.title(f'SHAP Summary Plot - {nombre_modelo}', fontsize=12, pad=15)
            plt.tight_layout()

            # Guardar gráfico
            if paths is not None:
                nombre_clean = nombre_modelo.replace(' ', '_').replace('(', '').replace(')', '')
                ruta_figura = os.path.join(paths['img'], f'shap_summary_{nombre_clean}.png')
                plt.savefig(ruta_figura, bbox_inches='tight', dpi=300)
                print(f"   Gráfico SHAP guardado en: {ruta_figura}")

            plt.show()
            plt.close()

        except Exception as e:
            print(f"   No se pudo generar el análisis SHAP para {nombre_modelo}: {e}")


# ============================================================
# 9. FUNCION PRINCIPAL
# ============================================================
def main():

    # Antes que nada Inicializamos las rutas del proyecto
    PATHS = inicializar_proyecto()

    # --------------------------------------------------------
    # Paso 1: Cargar dataset
    # --------------------------------------------------------

    ruta_archivo = 'student_data.csv'

    original = cargar_dataset(f"{PATHS['data']}/{ruta_archivo}")

    # Forma correcta: crea una copia independiente en memoria
    df = original.copy()

    if df is None:
        return
    print("\n==========================================")
    print("\n--- Información general del dataset ---")
    print("\n==========================================")
    print(df.info())

    # --------------------------------------------------------
    # Paso 2: Valores estadisticos del dataframe al inicio del proceso
    # --------------------------------------------------------

    print("\n==========================================")
    print("RESUMEN ESTADISTICOS ORIGINAL DEL DATAFRAME")
    print("==========================================")
    print(df.describe())

    # --------------------------------------------------------
    # Paso 3: Valores nulos
    # --------------------------------------------------------

    print("\n==========================================")
    print("REVISIÓN DE VALORES NULOS")
    print("==========================================")

    print(df.isnull().sum())

    # --------------------------------------------------------
    # Paso 4: Valores duplicados
    # --------------------------------------------------------

    print("\n==========================================")
    print("TRATAMIENTO DE DUPLICADOS")
    print("==========================================")

    duplicados = df.duplicated().sum()

    print(
        f"Cantidad de filas duplicadas exactas: "
        f"{duplicados}"
    )

    # --------------------------------------------------------
    # Paso 5: Valores atípicos (boxplots, z-score, IQR)
    # --------------------------------------------------------

    df, resultados_atipicos = identificar_valores_atipicos(df, PATHS)

    # --------------------------------------------------------
    # Paso 5.B: Consistencia de categorías
    # --------------------------------------------------------

    inconsistencias_categorias = revisar_consistencia_categorias(df)

    # --------------------------------------------------------
    # Paso 6: Variables categóricas binarias (yes/no, M/F, U/R, T/A)
    # --------------------------------------------------------

    df = convertir_variables_categoricas(df)

    # --------------------------------------------------------
    # Paso 7: Variables ordinales
    # --------------------------------------------------------

    df = codificar_variables_ordinales(df)

    # --------------------------------------------------------
    # Paso 8: Nuevas variables
    # --------------------------------------------------------

    df = crear_nuevas_variables(df)
    #---------------------------------------------------------------
    # Paso 1.1 Histograma + boxplot + skew/kurtosis del target (y opcionalmente otras variables)
    #---------------------------------------------------------------

    resultados_skew = analisis_univariado_target(df, PATHS)


    # --------------------------------------------------------
    # Paso 8.B: Análisis bivariado / multivariado
    #
    # --------------------------------------------------------

    resultados_bivariado = analisis_bivariado_multivariado(df, PATHS)

    # --------------------------------------------------------
    # Paso 8.C: Codificación de variables nominales (one-hot)
    #
    # Se hace recién acá, después del análisis bivariado, para no
    # perder los boxplots categóricos, pero ANTES de normalizar y
    # seleccionar variables, para que estas columnas sí compitan en
    # el SelectKBest
    # --------------------------------------------------------

    df = codificar_variables_nominales(df)

    # Opcional: si querés un VIF que también contemple las dummies
    # recién creadas, descomentar la siguiente línea (va a repetir
    # el heatmap/scatter, así que por defecto queda apagado):

    resultados_bivariado_dummies = analisis_bivariado_multivariado(df, PATHS)

    # --------------------------------------------------------
    # Paso 9: Normalización
    # --------------------------------------------------------

    df = normalizar_variables(df)

    # --------------------------------------------------------
    # Paso 10: Selección de variables relevantes
    # --------------------------------------------------------

    df, variables_seleccionadas, ranking = (seleccionar_variables_relevantes(df))

    # --------------------------------------------------------
    # Dataset final
    # --------------------------------------------------------

    print("\n==========================================")
    print("DATASET FINAL")
    print("==========================================")

    print(f"Filas: {df.shape[0]}")
    print(f"Columnas: {df.shape[1]}")


    # Guardar dataset procesado
    df.to_csv(f"{PATHS['outputs']}/student_data_wrangling.csv",index=False)

    print(
        "\nDataset procesado guardado como "
        "'student_data_wrangling.csv'"
    )

    # --- ENTRENAMIENTO Y COMPARACIÓN DE MODELOS ---
    resumen_modelos, modelos_entrenados, extras = entrenar_y_comparar_familias_modelos(df, PATHS)

    # --- NUEVA LLAMADA: ANÁLISIS EXPLICATIVO SHAP ---
    feature_sets = {
        'A': FEATURE_SET_A,
        'B': FEATURE_SET_B
    }
    analisis_explicabilidad_shap(modelos_entrenados, df, feature_sets, PATHS)

    # --- SELECCIÓN Y GUARDADO DEL MEJOR MODELO ---
    seleccion_mejor_modelo = entrenar_mejor_modelo_pipeline(df,original, PATHS)


# ============================================================
# EJECUCION DEL PROGRAMA
# ============================================================

if __name__ == "__main__":
    main()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
import joblib
import os
import joblib
import pandas as pd

# 1. Cargar las rutas del proyecto
PATHS = inicializar_proyecto()

# 2. Cargar el dataset procesado que guardó tu script previamente
ruta_df_procesado = os.path.join(PATHS['outputs'], 'student_data_wrangling.csv')
df = pd.read_csv(ruta_df_procesado)

ruta_modelo = '/content/drive/MyDrive/Trabajo Final Data2/outputs/modelo_lineal_nota_final.pkl'

# 3. Cargar artefacto
artefacto = joblib.load(ruta_modelo)

# Ejemplo de predicción directa sobre el dataframe de prueba:
predicciones = pipeline_cargado.predict(df)
print("Primeras 5 predicciones:", predicciones[:5])


 Proyecto configurado en: /content/drive/MyDrive/Trabajo Final Data2
Primeras 5 predicciones: [ 9.57589505  9.84100835  6.32092415 13.28515914 11.04142657]


In [ ]:
import os
import joblib
import pandas as pd
import numpy as np

# 1. Cargar el artefacto guardado con el Pipeline y sus columnas esperadas
ruta_modelo = os.path.join(PATHS['outputs'], 'modelo_lineal_nota_final.pkl')
artefacto = joblib.load(ruta_modelo)

pipeline_cargado = artefacto['pipeline']
features_esperadas = artefacto['features']

# Prueba con valores ya ajustados a la escala del modelo
nuevos_alumnos_escalados = pd.DataFrame([
    {
        'alumno': 'Alumno A (En riesgo)',
        'edad': 0.5,
        'tiempo_viaje_colegio': 0.6,
        'domicilio': 1,
        'genero': 0,
        'clases_particulares_pagadas': 0,
        'internet_en_casa': 0,
        'madre_trabaja_health': 0,
        'indice_riesgo_academico': 1.5,
        'nivel_educativo_parental': 0.2,
        'indice_riesgo_social': 1.0
    },
    {
        'alumno': 'Alumno B (Alto rendimiento)',
        'edad': 0.2,
        'tiempo_viaje_colegio': 0.1,
        'domicilio': 1,
        'genero': 1,
        'clases_particulares_pagadas': 1,
        'internet_en_casa': 1,
        'madre_trabaja_health': 1,
        'indice_riesgo_academico': -0.8,
        'nivel_educativo_parental': 0.9,
        'indice_riesgo_social': 0.2
    }
])

# 3. Realizar predicción con el Pipeline
predicciones_raw = pipeline_cargado.predict(nuevos_alumnos_escalados[features_esperadas])

# 4. Acotar predicciones dentro del rango válido (0 a 20)
predicciones_finales = np.clip(predicciones_raw, 0.0, 20.0)

# 5. Formatear y mostrar resultados
resultados = pd.DataFrame({
    'Alumno': nuevos_alumnos_escalados['alumno'],
    'Nota Final Predicha (0-20)': np.round(predicciones_finales, 2)
})

print("=== PREDICCIONES PARA NUEVOS CASOS DE PRUEBA ===")
print(resultados.to_string(index=False))

=== PREDICCIONES PARA NUEVOS CASOS DE PRUEBA ===
                     Alumno  Nota Final Predicha (0-20)
       Alumno A (En riesgo)                        4.85
Alumno B (Alto rendimiento)                       14.69
